In [78]:
import math
import torch
from torch.optim import AdamW
import transformers
from transformers import AutoTokenizer
from transformers import AutoModelForMaskedLM
from transformers import DataCollatorForLanguageModeling
from transformers import pipeline
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import get_scheduler
from tqdm.auto import tqdm

from transformers import TrainingArguments
from transformers import Trainer

# Inference

In [53]:
model_name= "distilbert-base-uncased"
tokenizer= AutoTokenizer.from_pretrained(model_name)
model= AutoModelForMaskedLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [30]:
text= "This is the trial [MASK]."

inputs= tokenizer(text, return_tensors="pt")
token_logits= model(**inputs).logits

mask_token_index= torch.where(inputs['input_ids'] == tokenizer.mask_token_id)[1]
mask_token_logit= token_logits[0, mask_token_index, :]

top_5_tokens= torch.topk(mask_token_logit, 5, dim=1).indices[0].tolist()
for i in top_5_tokens:
    print(text.replace(tokenizer.mask_token, tokenizer.decode([i])))

This is the trial schedule.
This is the trial procedure.
This is the trial summary.
This is the trial stage.
This is the trial phase.


In [6]:
text = ["This is the best [MASK].", "This is the worst place [MASK].", "The quick brown fox jumps over the lazy [MASK]."]

inputs = tokenizer(text, return_tensors="pt", padding= True, Truncation= True)
token_logits = model(**inputs).logits

# Find mask positions
mask_positions = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=False)
# print("\nMask positions:", mask_positions.tolist())
for batch_idx, token_idx in mask_positions:
    mask_token_logits = token_logits[batch_idx, token_idx, :]

    top_5_tokens = torch.topk(mask_token_logits, 5).indices.tolist()

    print(f"\nInput: {text[batch_idx]}")
    for tok in top_5_tokens:
        prediction = tokenizer.decode([tok]).strip()
        print(text[batch_idx].replace(tokenizer.mask_token, prediction))


Input: This is the best [MASK].
This is the best picture.
This is the best result.
This is the best score.
This is the best tournament.
This is the best edition.

Input: This is the worst place [MASK].
This is the worst place ever.
This is the worst place here.
This is the worst place overall.
This is the worst place today.
This is the worst place available.

Input: The quick brown fox jumps over the lazy [MASK].
The quick brown fox jumps over the lazy grass.
The quick brown fox jumps over the lazy rabbit.
The quick brown fox jumps over the lazy river.
The quick brown fox jumps over the lazy fox.
The quick brown fox jumps over the lazy pond.


# Fine-tuning

In [69]:
model_name= "distilbert-base-uncased"
tokenizer= AutoTokenizer.from_pretrained(model_name)
model= AutoModelForMaskedLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [54]:
imdb_dataset= load_dataset("imdb")
imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [55]:
def tokenize_function(examples):
    result = tokenizer(examples["text"])
    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result

tokenized_datasets = imdb_dataset.map(tokenize_function, batched=True, remove_columns=["text", "label"])
tokenized_datasets

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (720 > 512). Running this sequence through the model will result in indexing errors


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 50000
    })
})

In [56]:
chunk_size= 128

def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()} # to make chunks add all the text to a single list

    total_length = len(concatenated_examples[list(examples.keys())[0]])

    total_length = (total_length // chunk_size) * chunk_size # to handle the last chunk if smaller than chunk size
    result = {k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)] for k, t in concatenated_examples.items()}

    result["labels"] = result["input_ids"].copy()
    return result

lm_dataset= tokenized_datasets.map(group_texts, batched=True)
lm_dataset

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 61291
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 59904
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 122957
    })
})

In [57]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

In [58]:
# sample visualization
samples = [lm_dataset["train"][i] for i in range(2)]
for sample in samples:
    _ = sample.pop("word_ids")

for chunk in data_collator(samples)["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")


'>>> [CLS] i rented i am curious - yellow from my video store because of all the controversy that surrounded it when it was first released in 1967. i also heard that at first it was seized by [MASK]. s. customs if it ever tried to enter this [MASK], therefore being a fan of films considered " controversial " i really had [MASK] see this for [MASK].fold br / > < [MASK] [MASK] > duffy plot is centered around a young swedish drama student named desk who wants to [MASK] everything [MASK] can about life. in particular [MASK] wants to [MASK] her attentions to making some sort of documentary on [MASK] the average swede thought about certain [MASK] issues such'

'>>> as [MASK] vietnam war and race issues in the united states. in between asking politicians and ordinary denizens of stockholm [MASK] their [MASK] on politics, [MASK] has sex with her drama teacher, classmates, and [MASK] men. < br [MASK] > < br / > what kills meedd [MASK] am curious - yellow is that 40 [MASK] ago, this was conside

In [59]:
train_size = 10_000
test_size = int(0.1 * train_size)

downsampled_dataset = lm_dataset["train"].train_test_split(
    train_size=train_size, test_size=test_size, seed=42
)
downsampled_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 1000
    })
})

In [60]:
from huggingface_hub import notebook_login

notebook_login()

In [74]:
batch_size = 64
# Show the training loss with every epoch
logging_steps = len(downsampled_dataset["train"]) // batch_size
model_name = model_name.split("/")[-1]

training_args = TrainingArguments(
    # evaluation_strategy="epoch",
    # num_train_epochs= 4,
    learning_rate=2e-4,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    push_to_hub=False,
    fp16=True,
    logging_steps=logging_steps
)

In [75]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=downsampled_dataset["train"],
    eval_dataset=downsampled_dataset["test"],
    data_collator=data_collator,
    # tokenizer=tokenizer,
)

In [63]:
print("Before Training:")

eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Before Training:


>>> Perplexity: 21.94


In [76]:
trainer.train()

Step,Training Loss
156,2.557521
312,2.450806
468,2.389183


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=471, training_loss=2.465896075951319, metrics={'train_runtime': 151.0959, 'train_samples_per_second': 198.549, 'train_steps_per_second': 3.117, 'total_flos': 994208670720000.0, 'train_loss': 2.465896075951319, 'epoch': 3.0})

In [77]:
print("After Training:")

eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

After Training:


>>> Perplexity: 11.09


In [ ]:
text= "This is a trial [MASK]."

mask_filler= pipeline("fill-mask", model=model, tokenizer=tokenizer)
preds = mask_filler(text)

for pred in preds:
    print(f"{pred['sequence']}") #compare to inference without training

>>> this is a trial case.
>>> this is a trial film.
>>> this is a trial movie.
>>> this is a trial trial.
>>> this is a trial series.
